In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path('../data/raw/ml-latest-small')
OUT_DIR  = Path('../data/processed')
OUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

Device : cpu
PyTorch : 2.11.0+cpu


In [2]:
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')

# Filtre users et films peu actifs
user_counts  = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()
ratings = ratings[
    ratings['userId'].isin(user_counts[user_counts >= 20].index) &
    ratings['movieId'].isin(movie_counts[movie_counts >= 10].index)
].copy()

# Encodage continu 0..N-1 (requis pour les embeddings)
user2idx  = {u: i for i, u in enumerate(ratings['userId'].unique())}
movie2idx = {m: i for i, m in enumerate(ratings['movieId'].unique())}
idx2movie = {i: m for m, i in movie2idx.items()}

ratings['user_idx']  = ratings['userId'].map(user2idx)
ratings['movie_idx'] = ratings['movieId'].map(movie2idx)

N_USERS  = len(user2idx)
N_MOVIES = len(movie2idx)
print(f'Users  : {N_USERS:,} | Films : {N_MOVIES:,} | Ratings: {len(ratings):,}')
ratings.head()

Users  : 610 | Films : 2,269 | Ratings: 81,116


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1,4.0,2000-07-30 18:45:03,0,0
1,1,3,4.0,2000-07-30 18:20:47,0,1
2,1,6,4.0,2000-07-30 18:37:04,0,2
3,1,47,5.0,2000-07-30 19:03:35,0,3
4,1,50,5.0,2000-07-30 18:48:51,0,4


In [3]:
# Split chronologique — plus réaliste qu'un split aléatoire
ratings_sorted = ratings.sort_values('timestamp')
n = len(ratings_sorted)

train_df = ratings_sorted.iloc[:int(n * 0.8)].copy()
val_df   = ratings_sorted.iloc[int(n * 0.8):int(n * 0.9)].copy()
test_df  = ratings_sorted.iloc[int(n * 0.9):].copy()

# Filtre cold-start : garde users/films vus en train
train_users  = set(train_df['user_idx'])
train_movies = set(train_df['movie_idx'])
val_df  = val_df[val_df['user_idx'].isin(train_users)  & val_df['movie_idx'].isin(train_movies)]
test_df = test_df[test_df['user_idx'].isin(train_users) & test_df['movie_idx'].isin(train_movies)]

print(f'Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}')

Train : 64,892 | Val : 468 | Test : 486


In [4]:
class MovieDataset(Dataset):
    """Paires (user, item_pos, item_neg) pour la loss BPR."""
    def __init__(self, df, n_movies, n_neg=1):
        self.users    = torch.LongTensor(df['user_idx'].values)
        self.items    = torch.LongTensor(df['movie_idx'].values)
        self.ratings  = torch.FloatTensor(df['rating'].values)
        self.n_movies = n_movies
        # Films vus par chaque user → pour le negative sampling
        self.user_items = df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    def __len__(self): return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx].item()
        pos  = self.items[idx].item()
        neg  = np.random.randint(self.n_movies)
        while neg in self.user_items.get(user, set()):
            neg = np.random.randint(self.n_movies)
        return torch.tensor(user), torch.tensor(pos), torch.tensor(neg), self.ratings[idx]

train_loader = DataLoader(MovieDataset(train_df, N_MOVIES), batch_size=512, shuffle=True)
val_loader   = DataLoader(MovieDataset(val_df,   N_MOVIES), batch_size=512, shuffle=False)
test_loader  = DataLoader(MovieDataset(test_df,  N_MOVIES), batch_size=512, shuffle=False)

user, pos, neg, rating = next(iter(train_loader))
print(f'Batch OK — user={tuple(user.shape)} pos={tuple(pos.shape)} neg={tuple(neg.shape)}')

Batch OK — user=(512,) pos=(512,) neg=(512,)


In [5]:
# Sauvegarde splits + mappings
train_df.to_parquet(OUT_DIR / 'train.parquet', index=False)
val_df.to_parquet(OUT_DIR   / 'val.parquet',   index=False)
test_df.to_parquet(OUT_DIR  / 'test.parquet',  index=False)

meta = {
    'user2idx': user2idx, 'movie2idx': movie2idx,
    'idx2movie': idx2movie,
    'n_users': N_USERS,   'n_movies': N_MOVIES,
}
with open(OUT_DIR / 'meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('Fichiers sauvegardés :')
for p in sorted(OUT_DIR.glob('*')):
    print(f'  {p.name} ({p.stat().st_size/1024:.0f} KB)')
print('Prétraitement terminé — prêt pour les modèles !')

Fichiers sauvegardés :
  01_rating_distribution.png (122 KB)
  02_activity_distribution.png (36 KB)
  03_sparsity_matrix.png (123 KB)
  04_popularity_bias.png (260 KB)
  05_genres.png (55 KB)
  meta.pkl (79 KB)
  test.parquet (13 KB)
  train.parquet (749 KB)
  val.parquet (13 KB)
Prétraitement terminé — prêt pour les modèles !
